# clarify_test.ipynb — 要素澄清循环全链路验证 (7用例)

按 spec §8 验收表覆盖: ① CaseElements 模型单元 ② 单轮澄清 interrupt/resume ③ 3轮逐个补齐 ④ 5轮上限软退出 ⑤ 空回答=跳过 ⑥ 闲聊全 na 直通 ⑦ 高风险拒绝/确认两分支。全部节点行为真实, 仅 LLM 调用以测试替身接管(与 tests/test_smoke.py 同体系)。

## Cell 1 环境与测试替身 (setup, 自包含)

In [ ]:
import sys, asyncio, json
from pathlib import Path
from unittest.mock import patch

ROOT = Path.cwd().parent if Path.cwd().name == "lawApp_LangGraph" else Path.cwd()
sys.path.insert(0, str(ROOT))

# LLM 替身(移植 tests/test_smoke.py 体系)
from langchain_core.runnables import Runnable

class _FakeMsg:
    def __init__(self, content="", tool_calls=None):
        self.content = content
        self.tool_calls = tool_calls or []

class _FakeVerdict:
    def __init__(self, **kw):
        self.plan = list(kw.get("plan", ()))
        self.reasoning = list(kw.get("reasoning", ()))
        self.need_clarification = kw.get("need_clarification", False)
        self.question = kw.get("question", "")
        self.high_risk = kw.get("high_risk", False)
        self.needs_replan = kw.get("needs_replan", False)
        self.reason = kw.get("reason", "")
        self.applicable = kw.get("applicable", True)
        self.element_updates = list(kw.get("element_updates", ()))
        self.na_keys = list(kw.get("na_keys", ()))
        self.promote_keys = list(kw.get("promote_keys", ()))
        self.questions = list(kw.get("questions", ()))
        self.done = kw.get("done", False)
        self.insufficient_reason = kw.get("insufficient_reason", "none")

class _FakeChain(Runnable):
    def __init__(self, result=None):
        self.result = result
    def invoke(self, _inp, config=None, **kwargs):
        return self.result
    async def ainvoke(self, _inp, config=None, **kwargs):
        return self.result
    async def astream(self, _inp, config=None, **kwargs):
        yield _FakeMsg("测试回答")

class _FakeLLM(Runnable):
    def __init__(self, state):
        self.state = state
    def invoke(self, msgs, config=None, **kwargs):
        return self.state["executor_result"]
    def with_structured_output(self, schema):
        return _FakeChain(result=self.state["plan_result"])
    def bind_tools(self, tools):
        return self
    async def ainvoke(self, msgs, config=None, **kwargs):
        return self.state["executor_result"]
    async def astream(self, _prompt, config=None, **kwargs):
        yield _FakeMsg("测试回答")

import lawApp_LangGraph.LangGraph_lawApp as app
import lawApp_LangGraph.tools.rag_tools as rag_tools

class _ToolLLM:
    async def astream(self, _prompt):
        yield _FakeMsg("分析结果:测试回答")

_ctrl = {"plan_result": _FakeVerdict(), "executor_result": None}
_p_planner = patch.object(app, "get_planner_llm", lambda: _FakeLLM(_ctrl))
_p_executor = patch.object(app, "get_executor_llm", lambda: _FakeLLM(_ctrl))
_p_rag = patch.object(rag_tools, "_get_llm", lambda: _ToolLLM())
_p_planner.start(); _p_executor.start(); _p_rag.start()
print("setup ok")

## 用例① CaseElements 单元验证

In [ ]:
from lawApp_LangGraph.state import (
    CaseElements, default_case_elements,
)

ce = default_case_elements()
assert len(ce.elements) == 7
assert [e.key for e in ce.critical_missing()] == [
    "marriage_status", "demand", "property"]
ce.mark_na(["evidence"]); ce.promote(["timeline"])
ce.update("marriage_status", "在婚,分居中", by="ask")
assert ce.digest() == "婚姻现状:在婚,分居中"
ce.update("property", "一套房,双方名下", by="ask")
assert ce.digest() == "婚姻现状:在婚,分居中 | 主要财产与归属:一套房,双方名下"
print("① CaseElements model OK")

## 用例② 单轮澄清 → interrupt → resume → 要素齐直通

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command
from lawApp_LangGraph.state import ElementQuestion

def _q(key, q):
    return ElementQuestion(key=key, question=q)

async def case2():
    _ctrl["plan_result"] = _FakeVerdict(
        applicable=True, done=False,
        questions=[_q("marriage_status", "请问结婚几年了?现在是什么状态?")],
        plan=[],
    )
    g = app.build_graph(checkpointer=MemorySaver())
    cfg = {"configurable": {"thread_id": "nb-c2"}}
    r = await g.ainvoke({"query": "我想离婚"}, config=cfg)
    assert not r.get("final_answer")
    snap = await g.aget_state(cfg)
    intr = next(iter(snap.interrupts))
    assert intr.value["type"] == "clarify"
    assert intr.value["round"] == "1/5"
    assert len(intr.value["elements"]) == 7  # 面板载荷

    # resume → assess 第二轮 done=True → planner(空计划) → finalize
    _ctrl["plan_result"] = _FakeVerdict(applicable=True, done=True, plan=[])
    r2 = await g.ainvoke(Command(resume="结婚5年,分居中"), config=cfg)
    assert r2["final_answer"] == "测试回答"
    assert r2["clarify_rounds"] == 1
    assert r2["clarify_history"][0].answer == "结婚5年,分居中"
    print("② 单轮澄清 resume OK")

await case2()

## 用例③ 3轮逐个补齐 (assess 每轮吐一个反问, 第3轮 done)

In [ ]:
async def case3():
    g = app.build_graph(checkpointer=MemorySaver())
    cfg = {"configurable": {"thread_id": "nb-c3"}}
    rounds = [
        _FakeVerdict(applicable=True, done=False,
                     questions=[_q("marriage_status", "结婚几年了?")], plan=[]),
        _FakeVerdict(applicable=True, done=False,
                     element_updates=[type("U", (), {"key": "marriage_status",
                                                    "value": "5年", "status": "known"})()],
                     questions=[_q("demand", "你最想达到什么结果?")], plan=[]),
        _FakeVerdict(applicable=True, done=True, plan=[]),
    ]
    it = iter(rounds)
    _ctrl["plan_result"] = next(it)
    r = await g.ainvoke({"query": "我想离婚"}, config=cfg)
    for ans, expect_rounds in (("结婚5年", 1), ("想争取孩子抚养权", 2)):
        assert not r.get("final_answer")
        _ctrl["plan_result"] = next(it)
        r = await g.ainvoke(Command(resume=ans), config=cfg)
        assert r["clarify_rounds"] == expect_rounds
    assert r["final_answer"] == "测试回答"
    assert len(r["clarify_history"]) == 2
    print("③ 3轮逐个补齐 OK")

await case3()

## 用例④ 5轮上限软退出 (永远缺 → 第5轮 route_after_ask 强制放行)

In [ ]:
async def case4():
    g = app.build_graph(checkpointer=MemorySaver())
    cfg = {"configurable": {"thread_id": "nb-c4"}}
    # 永远吐一个反问(done=False)
    always_ask = _FakeVerdict(applicable=True, done=False,
                              questions=[_q("marriage_status", "结婚几年了?")],
                              plan=[])
    _ctrl["plan_result"] = always_ask
    r = await g.ainvoke({"query": "我想离婚"}, config=cfg)
    # 共 5 次 interrupt(1..5/5): 首次 + 前4次 resume 各触发新一轮反问
    for i in range(5):
        r = await g.ainvoke(Command(resume=f"回答{i}"), config=cfg)
    # 第5轮 resume 后轮数达上限 → route_after_ask 放行 → planner(空计划) → finalize
    assert r["clarify_rounds"] == 5
    assert r["final_answer"] == "测试回答"
    print("④ 5轮上限软退出 OK")

await case4()

## 用例⑤ 空回答=跳过按原问题继续 + 用例⑥ 闲聊全 na 直通

In [ ]:
async def case5():
    g = app.build_graph(checkpointer=MemorySaver())
    cfg = {"configurable": {"thread_id": "nb-c5"}}
    _ctrl["plan_result"] = _FakeVerdict(
        applicable=True, done=False,
        questions=[_q("marriage_status", "结婚几年了?")], plan=[])
    r = await g.ainvoke({"query": "我想离婚"}, config=cfg)
    _ctrl["plan_result"] = _FakeVerdict(applicable=True, done=True, plan=[])
    r2 = await g.ainvoke(Command(resume=""), config=cfg)  # 空回答
    assert r2["clarify_rounds"] == 5          # 置满 → 软放行
    assert "[用户补充信息]" not in r2["query"]  # 原问题未变
    assert r2["final_answer"] == "测试回答"
    print("⑤ 空回答跳过 OK")

async def case6():
    g = app.build_graph(checkpointer=MemorySaver())
    cfg = {"configurable": {"thread_id": "nb-c6"}}
    # applicable=false → 全 na 直通零反问
    _ctrl["plan_result"] = _FakeVerdict(applicable=False, plan=[])
    r = await g.ainvoke({"query": "今天天气怎么样"}, config=cfg)
    assert all(e.status == "na" for e in r["case_elements"].elements)
    assert r["clarify_rounds"] == 0
    assert r["final_answer"] == "测试回答"
    print("⑥ 闲聊全na直通 OK")

await case5(); await case6()

## 用例⑦ 高风险拒绝/确认两分支

In [ ]:
async def case7():
    g = app.build_graph(checkpointer=MemorySaver())
    # 拒绝分支
    cfg = {"configurable": {"thread_id": "nb-c7a"}}
    _ctrl["plan_result"] = _FakeVerdict(high_risk=True, plan=[])
    r = await g.ainvoke({"query": "我不想活了"}, config=cfg)
    snap = await g.aget_state(cfg)
    assert next(iter(snap.interrupts)).value["type"] == "risk_confirm"
    r2 = await g.ainvoke(Command(resume=False), config=cfg)   # normalize: False
    assert "12338" in r2["final_answer"]          # 热线文案
    assert r2["case_elements"].elements[0].status == "missing"  # 未进要素评估

    # 确认分支 → 继续要素评估
    cfg = {"configurable": {"thread_id": "nb-c7b"}}
    _ctrl["plan_result"] = _FakeVerdict(high_risk=True, applicable=True,
                                        done=True, plan=[])
    r = await g.ainvoke({"query": "家暴想离婚"}, config=cfg)
    r2 = await g.ainvoke(Command(resume=True), config=cfg)
    assert r2["risk_confirmed"] is True
    assert r2["final_answer"] == "测试回答"
    print("⑦ 高风险两分支 OK")

await case7()

## 收尾 teardown

In [ ]:
_p_planner.stop(); _p_executor.stop(); _p_rag.stop()
print("ALL PASSED")